# Pipeline ontology độc lập — 5 bước, code bằng hàm

Chỉ cần **notebook này + folder `ontology_pipeline/`**. Không import `src/`, không cần scripts/config của repo cũ.
Chạy từng cell từ trên xuống. Các cell định nghĩa hàm không gọi API; cell **Chạy bước N** mới xử lý.

| Bước | Đọc từ `input/` của bước | Ghi vào `output/` của bước |
|---|---|---|
| 1. Schema | SQLite local hoặc `schema.sql`; `metadata.json` | `schema_inventory.json`, báo cáo nguồn |
| 2. Base Ontology | Inventory; prompt; LLM Response Contract | Proposal, Class/Properties/Methods, trace |
| 3. Context | Inventory; toàn bộ Base Ontologies | Context FK trực tiếp từng Class |
| 4. Relations | Inventory; contexts; prompt; contract | Relations vật lý + diễn giải LLM, trace |
| 5. Kiểm định/xuất | Inventory; bases; proposals; contexts; relations | Ontology hợp nhất/từng Class, QC report, AI reviews |

Cell đầu mỗi bước **copy input cần thiết từ output bước trước** vào input bước hiện tại.
Sau đó hàm xử lý chỉ đọc thư mục bước đó. Không có kế thừa, class tự viết hoặc framework điều phối.
Một `try/except ValueError` duy nhất ở biên LLM để yêu cầu sửa JSON tối đa 1 lần; lỗi cuối được báo thẳng.

**LLM Response Contract** là JSON Schema mô tả hình dạng câu trả lời bắt buộc:
field nào cần có, kiểu dữ liệu, enum, giới hạn số lượng. Contract kiểm tra cấu trúc;
các hàm kiểm định bên dưới kiểm tra logic tham chiếu/cột/uniqueness/phép tổng hợp.
Contract không chứng minh toàn bộ ý nghĩa nghiệp vụ là đúng.


## Thiết lập và các hàm đọc/ghi chung

Cài các thư viện trong `ontology_pipeline/requirements.txt`, chọn Python kernel rồi chạy cell này.
`settings.json` chỉ chứa endpoint/model/thinking/timeout; API key lấy từ biến môi trường
`DC5_API_KEY` hoặc `ontology_pipeline/secrets.local.json` (copy từ secrets.example.json, không commit).
Kết nối mặc định trực tiếp; chỉ điền proxy_url nếu máy bạn cần proxy.
`FORCE=False` dùng cache theo input+prompt+contract+cấu hình; đổi prompt tự sinh lại phần bị ảnh hưởng.


In [1]:
from pathlib import Path
from contextlib import closing
from collections import Counter, defaultdict
from datetime import datetime, timezone
import copy
import hashlib
import html
import json
import os
import re
import shutil
import sqlite3
import time
import requests
from jsonschema import Draft202012Validator
from IPython.display import display, HTML

cwd = Path.cwd().resolve()
DATA = next((p / "ontology_pipeline" for p in (cwd, *cwd.parents)
             if (p / "ontology_pipeline/settings.json").is_file()), None)
if DATA is None:
    raise FileNotFoundError("Đặt notebook cạnh folder ontology_pipeline rồi mở lại.")
S1 = DATA / "01_schema"
S2 = DATA / "02_base_ontology"
S3 = DATA / "03_reference_context"
S4 = DATA / "04_relations"
S5 = DATA / "05_validation"
FORCE = False
RUN_AI_REVIEW = True

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\n"
    key = os.environ.get("DC5_API_KEY", "")
    secret_path = DATA / "secrets.local.json"
    if not key and secret_path.is_file():
        key = read_json(secret_path).get("api_key", "")
    if key:
        text = text.replace(key, "[REDACTED]")
    path.write_text(text, encoding="utf-8")

def fingerprint(value):
    text = json.dumps(value, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(text.encode()).hexdigest()

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else hashlib.sha256(stream.read()).hexdigest()

def require(condition, message):
    if not condition:
        raise ValueError(message)

def unique(values):
    return len(values) == len(set(values))

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))

def show_table(rows, columns):
    def cell(value):
        text = json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else str(value)
        return html.escape(text)
    head = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{cell(row.get(c, ''))}</td>" for c in columns)
                   + "</tr>" for row in rows)
    display(HTML(f"<div style='overflow:auto'><table><tr>{head}</tr>{body}</table></div>"))

def copy_file(source, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

def schema_errors(contract, value):
    return [f"{list(e.path)}: {e.message}" for e in Draft202012Validator(contract).iter_errors(value)]

settings = read_json(DATA / "settings.json")
print("Folder:", DATA)
show(settings)  # Không có API key trong settings.


Folder: /home/baond12/Documents/Auto_Build_Ontology_From_DB/database-base-ontology/ontology_pipeline
{
  "database_id": "omnichannel_retail",
  "base_url": "https://code-agent.cads.live/v1",
  "model": "qwen/qwen3.6-27b",
  "enable_thinking": true,
  "max_tokens": 16000,
  "timeout_seconds": 300,
  "proxy_url": ""
}


## Bước 1 — trích xuất schema

**Input:** `01_schema/input/database.local.sqlite3` nếu có; nếu không, dựng database rỗng trong RAM
từ `schema.sql`. `metadata.json` là mô tả nguồn tùy chọn, mặc định `{}`.
File SQL được đóng gói có cả CREATE TABLE và CREATE INDEX, không chứa rows.
**Output:** inventory có DDL, cột, kiểu, PK/FK, nullable, default, generated, UNIQUE và SQL của indexes.

Các hàm nhỏ bên dưới lần lượt đọc indexes, cột và FK. SQLite local mở read-only.
TEXT PRIMARY KEY trên rowid table có thể nullable; partial/expression UNIQUE không là khóa unique toàn bảng.


In [2]:
def quote_name(name):
    return '"' + name.replace('"', '""') + '"'

def read_indexes(connection, table_name):
    indexes = []
    for index in connection.execute(f"PRAGMA index_list({quote_name(table_name)})"):
        parts = connection.execute(f"PRAGMA index_xinfo({quote_name(index['name'])})").fetchall()
        definition = connection.execute("SELECT sql FROM sqlite_schema WHERE name=?", (index["name"],)).fetchone()
        indexes.append({
            "name": index["name"], "unique": bool(index["unique"]),
            "partial": bool(index["partial"]), "origin": index["origin"],
            "columns": [part["name"] for part in parts if part["key"]],
            "ddl": definition["sql"] if definition else None,
        })
    return sorted(indexes, key=lambda item: item["name"])

def read_columns(connection, table_name, flags, indexes, metadata):
    raw = connection.execute(f"PRAGMA table_xinfo({quote_name(table_name)})").fetchall()
    pk = [c["name"] for c in sorted(raw, key=lambda c: c["pk"]) if c["pk"]]
    rowid_alias = len(pk) == 1 and not any(i["origin"] == "pk" for i in indexes)
    require(not set(metadata) - {c["name"] for c in raw}, f"Metadata có cột không tồn tại: {table_name}")
    columns = []
    for column in raw:
        description = metadata.get(column["name"], {}).get("description", "")
        description = "" if description is None else description
        require(isinstance(description, str), f"Description phải là chuỗi: {table_name}.{column['name']}")
        implicit_not_null = column["pk"] and (
            flags["wr"] or flags["strict"] or (rowid_alias and column["type"].upper() == "INTEGER"))
        columns.append({
            "name": column["name"], "data_type": column["type"],
            "nullable": not bool(column["notnull"] or implicit_not_null),
            "primary_key_position": column["pk"], "default": column["dflt_value"],
            "generated": column["hidden"] in (2, 3), "description": description,
        })
    return columns, pk

def read_foreign_keys(connection, table_name):
    groups = defaultdict(list)
    for row in connection.execute(f"PRAGMA foreign_key_list({quote_name(table_name)})"):
        groups[row["id"]].append(dict(row))
    foreign_keys = []
    for parts in groups.values():
        parts.sort(key=lambda row: row["seq"])
        foreign_keys.append({
            "source_table": table_name, "source_columns": [p["from"] for p in parts],
            "target_table": parts[0]["table"], "target_columns": [p["to"] for p in parts],
            "on_update": parts[0]["on_update"], "on_delete": parts[0]["on_delete"], "match": parts[0]["match"],
        })
    return foreign_keys

def unique_keys(table):
    keys = [table["primary_key"]] if table["primary_key"] else []
    for index in table["indexes"]:
        if index["unique"] and not index["partial"] and None not in index["columns"]:
            keys.append(index["columns"])
    return keys

def has_unique_key(table, columns):
    return any(set(key) <= set(columns) for key in unique_keys(table))

def normalize_foreign_keys(tables):
    by_name = {table["name"]: table for table in tables}
    canonical = {name.lower(): name for name in by_name}
    for table in tables:
        source_names = {c["name"].lower(): c["name"] for c in table["columns"]}
        seen = Counter()
        for fk in table["foreign_keys"]:
            fk["target_table"] = canonical.get(fk["target_table"].lower(), fk["target_table"])
            require(fk["target_table"] in by_name, f"Thiếu bảng đích FK: {fk}")
            target = by_name[fk["target_table"]]
            target_names = {c["name"].lower(): c["name"] for c in target["columns"]}
            fk["source_columns"] = [source_names[name.lower()] for name in fk["source_columns"]]
            if all(name is None for name in fk["target_columns"]):
                fk["target_columns"] = target["primary_key"][:]
            else:
                require(all(name is not None for name in fk["target_columns"]), f"FK đích không đầy đủ: {fk}")
                fk["target_columns"] = [target_names[name.lower()] for name in fk["target_columns"]]
            require(len(fk["source_columns"]) == len(fk["target_columns"]), f"Số cột FK không khớp: {fk}")
            require(any(set(key) == set(fk["target_columns"]) for key in unique_keys(target)), f"Đích FK thiếu PK/UNIQUE: {fk}")
            signature = fingerprint(fk)[:24]
            seen[signature] += 1
            fk["relation_id"] = f"fk:{signature}:{seen[signature]}"
        table["foreign_keys"].sort(key=lambda fk: fk["relation_id"])
    return tables

def read_schema(connection, metadata):
    require(sqlite3.sqlite_version_info >= (3, 37), "Cần SQLite >= 3.37")
    require(isinstance(metadata, dict), "Metadata phải là JSON object")
    descriptions = metadata.get("tables", {})
    require(isinstance(descriptions, dict), "metadata.tables phải là object")
    connection.row_factory = sqlite3.Row
    definitions = connection.execute(
        "SELECT name,sql FROM sqlite_schema WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall()
    flags = {row["name"]: dict(row) for row in connection.execute("PRAGMA table_list")}
    require(not set(descriptions) - {row["name"] for row in definitions}, "Metadata có bảng không tồn tại")
    tables = []
    for definition in definitions:
        name = definition["name"]
        require(flags[name]["type"] == "table", f"Chưa hỗ trợ virtual/shadow table: {name}")
        info = descriptions.get(name, {})
        require(isinstance(info, dict) and isinstance(info.get("columns", {}), dict), f"Metadata sai cấu trúc: {name}")
        require(all(isinstance(item, dict) for item in info.get("columns", {}).values()), f"Metadata cột sai: {name}")
        description = info.get("description", "")
        description = "" if description is None else description
        require(isinstance(description, str), f"Description bảng phải là chuỗi: {name}")
        indexes = read_indexes(connection, name)
        columns, primary_key = read_columns(connection, name, flags[name], indexes, info.get("columns", {}))
        tables.append({"name": name, "description": description, "ddl": definition["sql"],
                       "columns": columns, "primary_key": primary_key, "indexes": indexes,
                       "foreign_keys": read_foreign_keys(connection, name)})
    violations = [dict(row) for row in connection.execute("PRAGMA foreign_key_check")]
    require(not violations, f"Database có {len(violations)} vi phạm FK")
    return normalize_foreign_keys(tables)

def extract_schema():
    sqlite_path = S1 / "input/database.local.sqlite3"
    schema_path = S1 / "input/schema.sql"
    metadata = read_json(S1 / "input/metadata.json")
    if sqlite_path.is_file():
        source = {"kind": "sqlite_read_only", "file": "input/database.local.sqlite3", "sha256": file_hash(sqlite_path)}
        connection = sqlite3.connect(sqlite_path.resolve().as_uri() + "?mode=ro", uri=True)
    else:
        source = {"kind": "schema_sql_in_memory", "file": "input/schema.sql", "sha256": file_hash(schema_path)}
        connection = sqlite3.connect(":memory:")
        connection.executescript(schema_path.read_text(encoding="utf-8"))
    with closing(connection):
        tables = read_schema(connection, metadata)
    if sqlite_path.is_file():
        require(file_hash(sqlite_path) == source["sha256"], "Database nguồn đã thay đổi khi đọc")
    payload = {"database_id": settings["database_id"], "tables": tables}
    inventory = {**payload, "schema_fingerprint": fingerprint(payload), "source": source}
    write_json(S1 / "output/schema_inventory.json", inventory)
    report = {"tables": len(tables), "columns": sum(len(t["columns"]) for t in tables),
              "foreign_keys": sum(len(t["foreign_keys"]) for t in tables), "source": source}
    write_json(S1 / "output/extraction_report.json", report)
    return report


### Chạy bước 1 và quan sát output


In [3]:
show(extract_schema())
schema = read_json(S1 / "output/schema_inventory.json")
products_schema = next(t for t in schema["tables"] if t["name"] == "products")
show_table(products_schema["columns"], ["name", "data_type", "nullable", "primary_key_position", "description"])


{
  "tables": 20,
  "columns": 157,
  "foreign_keys": 31,
  "source": {
    "kind": "sqlite_read_only",
    "file": "input/database.local.sqlite3",
    "sha256": "948e4a1bd0b5bbb4c20e75ccfea6c4e3422c033fa44575ad615d0c3156e36916"
  }
}


name,data_type,nullable,primary_key_position,description
product_id,VARCHAR(50),False,1,
sku,VARCHAR(100),False,0,
name,VARCHAR(255),False,0,
product_url,TEXT,True,0,
brand_id,INTEGER,True,0,
category_id,INTEGER,True,0,
promotion_id,INTEGER,True,0,
original_price,"NUMERIC(12,2)",True,0,
discount_price,"NUMERIC(12,2)",True,0,
currency,VARCHAR(10),True,0,


## Bước 2 — Base Ontology từng bảng

**Input:** inventory copy vào `02_base_ontology/input/`, `prompt.txt`, `response_contract.json`.
**Output:** `output/proposals/` (JSON LLM), `output/base_ontologies/` (JSON đã ghép), `output/traces/` (request, content, thinking, usage).

Contract mới ghi rõ operator của filter, group_by và phép aggregate. Nhờ đó có thể kiểm tra
Method mô tả count/sum/group theo đúng kế hoạch; lookup phải lọc equality bằng khóa unique đầy đủ.
`source_columns` phải đúng bằng hợp các cột thực sự dùng. Không có SQL runtime hoặc JOIN.

`input/review_feedback.json` chứa hướng dẫn sửa riêng từng bảng đã đối chiếu schema; được nối vào prompt chỉ cho bảng tương ứng. Đổi feedback chỉ sinh lại bảng đó và phần phụ thuộc.


In [4]:
def numeric_column(column):
    declared = column["data_type"].upper()
    return any(word in declared for word in ("INT", "REAL", "FLOA", "DOUB", "NUMERIC", "DECIMAL"))

def parameter_type(column):
    declared = column["data_type"].upper()
    if "BOOL" in declared:
        return "boolean"
    if "INT" in declared:
        return "integer"
    if numeric_column(column):
        return "number"
    return "string"

def method_errors(method, table):
    errors = []
    columns = {c["name"]: c for c in table["columns"]}
    params = method["input_parameters"]
    output = method["output"]
    groups = method["group_by"]
    metrics = output["aggregations"]
    used = set(output["source_columns"]) | set(groups)
    used.update(p["source_column"] for p in params)
    used.update(m["source_column"] for m in metrics if m["source_column"] is not None)
    if used != set(method["source_columns"]):
        errors.append("source_columns phải bằng hợp chính xác các cột input/output/group/aggregate")
    if not used <= set(columns):
        return errors + [f"Tham chiếu cột không tồn tại hoặc ngoài bảng: {sorted(used - set(columns))}"]
    if not unique([p["name"] for p in params]):
        errors.append("Tên input parameter bị trùng")
    for param in params:
        column = columns[param["source_column"]]
        if param["type"] != parameter_type(column):
            errors.append(f"Sai type của parameter {param['name']}")
        if any(word in column["data_type"].upper() for word in ("JSON", "BLOB")):
            errors.append("Chưa hỗ trợ filter trực tiếp JSON/BLOB")
        if param["operator"] == "contains" and param["type"] != "string":
            errors.append("contains chỉ dùng với string")
    if not unique([m["name"] for m in metrics] + groups):
        errors.append("Tên aggregation trùng nhau hoặc trùng cột group_by")
    for metric in metrics:
        name = metric["source_column"]
        if name is None and metric["function"] != "count":
            errors.append("Chỉ count(*) được dùng source_column=null")
        if name and metric["function"] in ("sum", "avg") and not numeric_column(columns[name]):
            errors.append("sum/avg chỉ dùng cột số")
    operation = method["operation"]
    if operation in ("list", "lookup"):
        if groups or metrics or not output["source_columns"]:
            errors.append("list/lookup phải trả cột thật, không group/aggregate")
        expected_kind = "rows" if operation == "list" else "object"
        if output["kind"] != expected_kind:
            errors.append(f"{operation} phải trả {expected_kind}")
    if operation == "lookup":
        equal_required = {p["source_column"] for p in params if p["required"] and p["operator"] == "eq"}
        if not has_unique_key(table, equal_required):
            errors.append("lookup phải equality-filter đầy đủ PK/UNIQUE bằng parameter required")
    if operation == "count":
        if (groups or output["kind"] != "scalar" or output["source_columns"] or len(metrics) != 1
                or metrics[0]["function"] != "count" or metrics[0]["source_column"] is not None):
            errors.append("count phải là COUNT(*) scalar, không group, không trả cột thô")
    if operation == "aggregate":
        if not metrics:
            errors.append("aggregate phải khai báo phép tổng hợp")
        if groups:
            if output["kind"] != "rows" or set(output["source_columns"]) != set(groups):
                errors.append("aggregate có nhóm phải trả rows và đủ đúng các cột nhóm")
        elif output["kind"] != "scalar" or output["source_columns"] or len(metrics) != 1:
            errors.append("aggregate không nhóm phải trả đúng một scalar metric")
    return errors

def base_proposal_errors(proposal, table):
    errors = []  # Contract đã được kiểm tra trước khi gọi hàm này.
    properties = proposal["properties"]
    names = [p["column_name"] for p in properties]
    if not unique(names) or set(names) != {c["name"] for c in table["columns"]}:
        errors.append("Properties phải bao phủ mỗi cột đúng một lần")
    self_fk_columns = {c for fk in table["foreign_keys"] if fk["target_table"] == table["name"]
                       for c in fk["source_columns"]}
    self_descriptions = [proposal["description"]] + [p["description"] for p in properties if p["column_name"] in self_fk_columns]
    if self_fk_columns and any(re.search(r"\bkhác\b", text.lower()) for text in self_descriptions):
        errors.append("Mô tả self-FK dùng 'trong cùng bảng', tránh 'khác' gây hiểu bắt buộc khác bản ghi")
    if not unique([p["name"] for p in properties]):
        errors.append("Tên Properties bị trùng")
    descriptions = [proposal["description"]] + [p["description"] for p in properties] + proposal["assumptions"]
    if "AUTOINCREMENT" not in table["ddl"].upper():
        if any(phrase in text.lower() for text in descriptions
               for phrase in ("tự động tăng", "tự tăng", "tăng theo thứ tự chèn")):
            errors.append("Không dùng khẳng định tự tăng khi DDL không khai báo AUTOINCREMENT; mô tả là khóa chính INTEGER")
    if not unique([m["name"] for m in proposal["methods"]]):
        errors.append("Tên Methods bị trùng")
    plans = []
    for method in proposal["methods"]:
        errors.extend(f"{method['name']}: {error}" for error in method_errors(method, table))
        plan = {key: method[key] for key in ("operation", "input_parameters", "group_by", "output")}
        # Parameter/metric display names do not distinguish the underlying query plan.
        plan["input_parameters"] = sorted(
            [{k: v for k, v in p.items() if k != "name"} for p in plan["input_parameters"]],
            key=lambda p: (p["source_column"], p["operator"], p["required"]))
        plan["output"] = copy.deepcopy(plan["output"])
        plan["output"]["source_columns"] = sorted(plan["output"]["source_columns"])
        plan["group_by"] = sorted(plan["group_by"])
        plan["output"]["aggregations"] = [
            {k: v for k, v in a.items() if k != "name"} for a in plan["output"]["aggregations"]]
        plan["output"]["aggregations"].sort(key=lambda a: (a["function"], a["source_column"] or ""))
        plans.append(fingerprint(plan))
    if not unique(plans):
        errors.append("Methods có kế hoạch trùng nhau, chỉ khác tên")
    return errors


### Hàm gọi LLM dùng chung cho bước 2, 4 và AI review ở bước 5

Đọc prompt/contract từ input của chính bước. Cache theo fingerprint; lưu request trước khi gửi;
lưu cả `reasoning_content` nếu server trả về.
Gửi `response_format=json_schema` với strict=True. Riêng Relations, contract còn gắn đúng số FK
và enum relation IDs cho request hiện tại. Validator vẫn kiểm tra đủ từng FK vì JSON đúng chưa đủ đúng logic. Timeout/HTTP lỗi được báo thẳng.
JSON sai được gửi lại kèm lỗi để sửa tối đa 1 lần, không bịa kết quả thay LLM.

DC5/xgrammar hiện không hỗ trợ keyword `uniqueItems`. Chỉ bản schema gửi API bỏ keyword này; validator local vẫn dùng contract đầy đủ và bắt trùng lặp.


In [5]:
def generation_contract(value):
    # DC5/xgrammar không hỗ trợ uniqueItems; giữ kiểm tra này ở validator local.
    if isinstance(value, dict):
        return {key: generation_contract(item) for key, item in value.items() if key != "uniqueItems"}
    if isinstance(value, list):
        return [generation_contract(item) for item in value]
    return value

def parse_response(content):
    content = content.strip()
    if content.startswith("```") and content.endswith("```"):
        content = content.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(content)

def api_key():
    key = os.environ.get("DC5_API_KEY", "")
    if not key and (DATA / "secrets.local.json").is_file():
        key = read_json(DATA / "secrets.local.json").get("api_key", "")
    require(bool(key) and key != "SET_YOUR_DC5_API_KEY_HERE", "Cần DC5_API_KEY hoặc secrets.local.json")
    return key

def ask_llm(step, name, context, check):
    prompt = (step / "input/prompt.txt").read_text(encoding="utf-8")
    feedback_path = step / "input/review_feedback.json"
    if feedback_path.is_file():
        feedback = read_json(feedback_path).get(name, "")
        if feedback:
            prompt += "\nLƯU Ý REVIEW ĐÃ ĐỐI CHIẾU SCHEMA:\n" + feedback
    contract = read_json(step / "input/response_contract.json")
    if step == S4:
        expected_ids = [fk["relation_id"] for fk in context["owned_foreign_keys"]]
        relation_schema = contract["properties"]["relations"]
        relation_schema.update(minItems=len(expected_ids), maxItems=len(expected_ids))
        relation_schema["items"]["properties"]["relation_id"]["enum"] = expected_ids
    generation = {key: settings[key] for key in ("base_url", "model", "enable_thinking", "max_tokens")}
    generation["structured_output"] = "json_schema"
    request_hash = fingerprint({"context": context, "prompt": prompt, "contract": contract, "generation": generation})
    trace_path = step / "output/traces" / f"{name}.json"
    if trace_path.is_file() and not FORCE:
        cached = read_json(trace_path)
        if cached.get("request_fingerprint") == request_hash and cached.get("status") == "COMPLETE":
            result = cached["proposal"]
            errors = schema_errors(contract, result)
            if not errors:
                errors = check(result)
            require(not errors, " | ".join(errors))
            print("CACHED", step.name, name, flush=True)
            return result
    messages = [{"role": "system", "content": prompt}, {"role": "user", "content": json.dumps(
        {"response_contract": contract, "context": context}, ensure_ascii=False)}]
    trace = {"request_fingerprint": request_hash, "generation": generation, "attempts": [], "status": "RUNNING"}
    print("CALL", step.name, name, flush=True)
    with requests.Session() as session:
        session.trust_env = False
        proxy = settings.get("proxy_url", "")
        proxies = {"http": proxy, "https": proxy} if proxy else {}
        for attempt in (1, 2):
            payload = {"model": settings["model"], "temperature": 0,
                       "max_tokens": settings["max_tokens"], "messages": messages,
                       "chat_template_kwargs": {"enable_thinking": settings["enable_thinking"]},
                       "response_format": {"type": "json_schema", "json_schema": {
                           "name": "ontology_response", "strict": True, "schema": generation_contract(contract)}}}
            trace["request"] = payload
            write_json(trace_path, trace)
            started = time.monotonic()
            response = session.post(settings["base_url"].rstrip("/") + "/chat/completions",
                                    headers={"Authorization": "Bearer " + api_key()}, json=payload,
                                    proxies=proxies, timeout=(15, settings["timeout_seconds"]))
            record = {"attempt": attempt, "http_status": response.status_code,
                      "elapsed_seconds": round(time.monotonic() - started, 3)}
            trace["attempts"].append(record)
            if not response.ok:
                record["error_body"] = response.text[:2000]
                trace["status"] = "HTTP_ERROR"
                write_json(trace_path, trace)
                response.raise_for_status()
            body = response.json()
            choice = body["choices"][0]
            message = choice["message"]
            record.update({"content": message.get("content") or "",
                           "reasoning_content": message.get("reasoning_content") or "",
                           "finish_reason": choice.get("finish_reason"), "usage": body.get("usage", {}),
                           "model": body.get("model"), "response_id": body.get("id")})
            write_json(trace_path, trace)
            try:
                require(choice.get("finish_reason") != "length", "Response bị cắt: tăng max_tokens")
                result = parse_response(record["content"])
                errors = schema_errors(contract, result)
                if not errors:
                    errors = check(result)
                require(not errors, " | ".join(errors))
            except ValueError as error:
                record["validation_error"] = str(error)
                trace["status"] = "INVALID_RESPONSE"
                write_json(trace_path, trace)
                if attempt == 2:
                    raise
                messages = messages + [{"role": "assistant", "content": record["content"]},
                                       {"role": "user", "content": "Sửa JSON theo input/contract. Lỗi: " + str(error)}]
                continue
            trace.update(status="COMPLETE", proposal=result)
            write_json(trace_path, trace)
            return result


In [6]:
def assemble_base(table, proposal, database_id):
    name = table["name"]
    annotations = {p["column_name"]: p for p in proposal["properties"]}
    fk_columns = {c for fk in table["foreign_keys"] for c in fk["source_columns"]}
    properties = []
    for column in table["columns"]:
        annotation = annotations[column["name"]]
        properties.append({
            "property_id": f"property:{name}.{column['name']}", "name": annotation["name"],
            "description": annotation["description"], "confidence": annotation["confidence"],
            "source_table": name, "source_column": column["name"],
            **{key: column[key] for key in ("data_type", "nullable", "primary_key_position", "default", "generated")},
            "is_foreign_key": column["name"] in fk_columns, "metadata_description": column["description"],
        })
    methods = []
    for method in proposal["methods"]:
        methods.append({**method, "method_id": f"method:{name}.{method['name']}",
                        "owner_class_id": f"class:{name}", "source_table": name,
                        "scope": "BASE_CLASS", "status": "PROPOSED_READ_ONLY"})
    return {
        "database_id": database_id,
        "class": {"class_id": f"class:{name}", "name": proposal["class_name"], "source_table": name,
                  "description": proposal["description"], "metadata_description": table["description"],
                  "confidence": proposal["confidence"], "semantic_status": "PROPOSED"},
        "properties": properties, "methods": methods, "assumptions": proposal["assumptions"],
        "input_fingerprint": fingerprint({"database_id": database_id, "table": table}),
    }

def build_bases():
    inventory = read_json(S2 / "input/schema_inventory.json")
    bases = {}
    for table in inventory["tables"]:
        name = table["name"]
        context = {"database_id": inventory["database_id"], "table": table}
        proposal = ask_llm(S2, name, context, lambda result: base_proposal_errors(result, table))
        base = assemble_base(table, proposal, inventory["database_id"])
        write_json(S2 / "output/proposals" / f"{name}.json", proposal)
        write_json(S2 / "output/base_ontologies" / f"{name}.json", base)
        bases[name] = base
    require(unique([b["class"]["name"] for b in bases.values()]), "Tên Class bị trùng giữa các bảng")
    return {"classes": len(bases), "properties": sum(len(b["properties"]) for b in bases.values()),
            "methods": sum(len(b["methods"]) for b in bases.values())}


### Chuẩn bị input và chạy bước 2 — toàn bộ bảng, DC5 thinking

Cell này gọi API thật khi không có cache. Xem file `input/prompt.txt` và `input/response_contract.json`
trước khi chạy. `settings.enable_thinking` được gửi dưới `chat_template_kwargs`.


In [7]:
copy_file(S1 / "output/schema_inventory.json", S2 / "input/schema_inventory.json")
show(build_bases())
product_base = read_json(S2 / "output/base_ontologies/products.json")
show(product_base["class"])
show_table(product_base["methods"], ["name", "operation", "input_parameters", "group_by", "output"])


CACHED 02_base_ontology addresses


CACHED 02_base_ontology agents


CACHED 02_base_ontology brands


CALL 02_base_ontology categories


CACHED 02_base_ontology channels


...

CACHED 02_base_ontology stores


CACHED 02_base_ontology suppliers


CACHED 02_base_ontology support_tickets


{
  "classes": 20,
  "properties": 157,
  "methods": 82
}
{
  "class_id": "class:products",
  "name": "Product",
  "source_table": "products",
  "description": "Sản phẩm trong hệ thống bán lẻ đa kênh, bao gồm thông tin định danh, giá, tồn kho, tham chiếu thương hiệu/danh mục/khuyến mãi và các trường dữ liệu mở rộng dạng JSONB.",
  "metadata_description": "",
  "confidence": 0.92,
  "semantic_status": "PROPOSED"
}


name,operation,input_parameters,group_by,output
lookupByProductId,lookup,"[{""name"": ""productId"", ""type"": ""string"", ""required"": true, ""source_column"": ""product_id"", ""operator"": ""eq""}]",[],"{""kind"": ""object"", ""source_columns"": [""product_id"", ""sku"", ""name"", ""product_url"", ""brand_id"", ""category_id"", ""promotion_id"", ""original_price"", ""discount_price"", ""currency"", ""discount_percentage"", ""is_in_stock"", ""stock_quantity"", ""delivery_locations"", ""specifications"", ""warranty"", ""images"", ""created_at"", ""description"", ""description_embedding""], ""aggregations"": []}"
lookupBySku,lookup,"[{""name"": ""sku"", ""type"": ""string"", ""required"": true, ""source_column"": ""sku"", ""operator"": ""eq""}]",[],"{""kind"": ""object"", ""source_columns"": [""product_id"", ""sku"", ""name"", ""product_url"", ""brand_id"", ""category_id"", ""promotion_id"", ""original_price"", ""discount_price"", ""currency"", ""discount_percentage"", ""is_in_stock"", ""stock_quantity"", ""delivery_locations"", ""specifications"", ""warranty"", ""images"", ""created_at"", ""description"", ""description_embedding""], ""aggregations"": []}"
listProducts,list,"[{""name"": ""brandId"", ""type"": ""integer"", ""required"": false, ""source_column"": ""brand_id"", ""operator"": ""eq""}, {""name"": ""categoryId"", ""type"": ""integer"", ""required"": false, ""source_column"": ""category_id"", ""operator"": ""eq""}, {""name"": ""isInStock"", ""type"": ""boolean"", ""required"": false, ""source_column"": ""is_in_stock"", ""operator"": ""eq""}, {""name"": ""minDiscountPrice"", ""type"": ""number"", ""required"": false, ""source_column"": ""discount_price"", ""operator"": ""gte""}, {""name"": ""maxDiscountPrice"", ""type"": ""number"", ""required"": false, ""source_column"": ""discount_price"", ""operator"": ""lte""}]",[],"{""kind"": ""rows"", ""source_columns"": [""product_id"", ""sku"", ""name"", ""product_url"", ""brand_id"", ""category_id"", ""promotion_id"", ""original_price"", ""discount_price"", ""currency"", ""discount_percentage"", ""is_in_stock"", ""stock_quantity"", ""delivery_locations"", ""specifications"", ""warranty"", ""images"", ""created_at"", ""description"", ""description_embedding""], ""aggregations"": []}"
countProducts,count,"[{""name"": ""brandId"", ""type"": ""integer"", ""required"": false, ""source_column"": ""brand_id"", ""operator"": ""eq""}, {""name"": ""categoryId"", ""type"": ""integer"", ""required"": false, ""source_column"": ""category_id"", ""operator"": ""eq""}, {""name"": ""isInStock"", ""type"": ""boolean"", ""required"": false, ""source_column"": ""is_in_stock"", ""operator"": ""eq""}]",[],"{""kind"": ""scalar"", ""source_columns"": [], ""aggregations"": [{""name"": ""rowCount"", ""function"": ""count"", ""source_column"": null}]}"
avgDiscountPriceByCategory,aggregate,"[{""name"": ""categoryId"", ""type"": ""integer"", ""required"": false, ""source_column"": ""category_id"", ""operator"": ""eq""}]","[""category_id""]","{""kind"": ""rows"", ""source_columns"": [""category_id""], ""aggregations"": [{""name"": ""avgDiscountPrice"", ""function"": ""avg"", ""source_column"": ""discount_price""}]}"


## Bước 3 — context tham chiếu một bước

**Input:** inventory và tất cả Base Ontologies copy vào input bước 3.
**Output:** context của từng Class gồm Class/Properties hiện tại và các Class liên kết FK trực tiếp,
cả chiều vào/ra và self-FK. Không gọi LLM, không đệ quy.
Context dành cho Relations nên không mang theo Methods để giảm token và tránh nhầm owner.


In [8]:
def ontology_view(base):
    return {"class": base["class"], "properties": base["properties"],
            "input_fingerprint": base["input_fingerprint"]}

def make_context(table_name, inventory, bases):
    foreign_keys = [fk for t in inventory["tables"] for fk in t["foreign_keys"]
                    if table_name in (fk["source_table"], fk["target_table"])]
    neighbors = sorted({name for fk in foreign_keys
                        for name in (fk["source_table"], fk["target_table"])} - {table_name})
    payload = {
        "database_id": inventory["database_id"], "class_id": f"class:{table_name}",
        "current": ontology_view(bases[table_name]),
        "references": [{"class_id": f"class:{name}", "ontology": ontology_view(bases[name])} for name in neighbors],
        "foreign_keys": foreign_keys,
    }
    return {**payload, "context_fingerprint": fingerprint(payload)}

def build_contexts():
    inventory = read_json(S3 / "input/schema_inventory.json")
    bases = {t["name"]: read_json(S3 / "input/base_ontologies" / f"{t['name']}.json")
             for t in inventory["tables"]}
    for table in inventory["tables"]:
        expected_hash = fingerprint({"database_id": inventory["database_id"], "table": table})
        require(bases[table["name"]]["input_fingerprint"] == expected_hash,
                f"Base ontology cũ: {table['name']}. Chạy lại bước 2.")
    rows = []
    for table in inventory["tables"]:
        context = make_context(table["name"], inventory, bases)
        write_json(S3 / "output/reference_contexts" / f"{table['name']}.json", context)
        rows.append({"table": table["name"], "references": len(context["references"]),
                     "foreign_keys": len(context["foreign_keys"])})
    return rows


In [9]:
copy_file(S1 / "output/schema_inventory.json", S3 / "input/schema_inventory.json")
for table in read_json(S3 / "input/schema_inventory.json")["tables"]:
    name = table["name"]
    copy_file(S2 / "output/base_ontologies" / f"{name}.json", S3 / "input/base_ontologies" / f"{name}.json")
show_table(build_contexts(), ["table", "references", "foreign_keys"])
product_context = read_json(S3 / "output/reference_contexts/products.json")
show({"class_id": product_context["class_id"], "references": [r["class_id"] for r in product_context["references"]]})


table,references,foreign_keys
addresses,2,2
agents,3,3
brands,1,1
categories,1,2
channels,2,2
chats,5,5
customers,5,5
inventory_snapshots,2,2
order_items,4,4
orders,9,9


{
  "class_id": "class:products",
  "references": [
    "class:brands",
    "class:categories",
    "class:chats",
    "class:inventory_snapshots",
    "class:order_items",
    "class:product_suppliers",
    "class:promotions"
  ]
}


## Bước 4 — Relations

**Input:** inventory, contexts, prompt và response contract trong input bước 4.
**Output:** `relations.json`, các annotations và traces.
Code giữ endpoints/cardinality. LLM chỉ mô tả đúng các FK mà bảng hiện tại sở hữu.
Một FK được sinh một lần; bước 5 gắn Relation vào cả hai Class, self-FK vẫn chỉ một Relation.


In [10]:
def physical_relation(fk, tables):
    source = tables[fk["source_table"]]
    nullable = {c["name"]: c["nullable"] for c in source["columns"]}
    return {
        "relation_id": fk["relation_id"],
        "source_class_id": f"class:{fk['source_table']}", "target_class_id": f"class:{fk['target_table']}",
        "source_property_ids": [f"property:{fk['source_table']}.{c}" for c in fk["source_columns"]],
        "target_property_ids": [f"property:{fk['target_table']}.{c}" for c in fk["target_columns"]],
        "cardinality": {
            "source_to_target": {"min": 0 if any(nullable[c] for c in fk["source_columns"]) else 1, "max": 1},
            "target_to_source": {"min": 0, "max": 1 if has_unique_key(source, fk["source_columns"]) else "many"},
        },
        "source_fk": fk, "physical_status": "VERIFIED_FROM_SCHEMA", "semantic_status": "PROPOSED",
    }

def relation_proposal_errors(proposal, foreign_keys):
    errors = []  # Contract đã được kiểm tra trước khi gọi hàm này.
    ids = [r["relation_id"] for r in proposal["relations"]]
    if not unique(ids) or set(ids) != {fk["relation_id"] for fk in foreign_keys}:
        errors.append(f"Mỗi FK đúng một Relation. Expected IDs={sorted(fk['relation_id'] for fk in foreign_keys)}; actual={ids}")
    return errors

def build_relations():
    inventory = read_json(S4 / "input/schema_inventory.json")
    tables = {t["name"]: t for t in inventory["tables"]}
    relations = []
    for name, table in tables.items():
        owned = table["foreign_keys"]
        if not owned:
            continue
        context = read_json(S4 / "input/reference_contexts" / f"{name}.json")
        payload = {key: value for key, value in context.items() if key != "context_fingerprint"}
        require(context["context_fingerprint"] == fingerprint(payload), f"Context bị sửa: {name}")
        expected_fks = [fk for t in tables.values() for fk in t["foreign_keys"]
                        if name in (fk["source_table"], fk["target_table"])]
        require(context["foreign_keys"] == expected_fks, f"Context có FK cũ: {name}")
        views = [context["current"]] + [r["ontology"] for r in context["references"]]
        for view in views:
            source_name = view["class"]["source_table"]
            expected_hash = fingerprint({"database_id": inventory["database_id"], "table": tables[source_name]})
            require(view["input_fingerprint"] == expected_hash, f"Context có ontology cũ: {source_name}")
        request = {"reference_context": context, "owned_foreign_keys": owned,
                   "physical_relations": [physical_relation(fk, tables) for fk in owned]}
        proposal = ask_llm(S4, name, request, lambda result: relation_proposal_errors(result, owned))
        write_json(S4 / "output/annotations" / f"{name}.json", proposal)
        annotations = {row["relation_id"]: row for row in proposal["relations"]}
        for fk in owned:
            relations.append({**annotations[fk["relation_id"]], **physical_relation(fk, tables)})
    require(unique([r["relation_id"] for r in relations]), "Relation IDs bị trùng")
    result = {"database_id": inventory["database_id"], "schema_fingerprint": inventory["schema_fingerprint"],
              "relations": sorted(relations, key=lambda r: r["relation_id"])}
    write_json(S4 / "output/relations.json", result)
    return {"relations": len(relations)}


In [11]:
copy_file(S1 / "output/schema_inventory.json", S4 / "input/schema_inventory.json")
for table in read_json(S4 / "input/schema_inventory.json")["tables"]:
    name = table["name"]
    copy_file(S3 / "output/reference_contexts" / f"{name}.json", S4 / "input/reference_contexts" / f"{name}.json")
show(build_relations())
relations_preview = read_json(S4 / "output/relations.json")["relations"]
show_table(relations_preview, ["name", "source_class_id", "target_class_id", "cardinality", "description"])


CACHED 04_relations addresses


CACHED 04_relations agents


CALL 04_relations categories


CACHED 04_relations chats


CACHED 04_relations customers


...

CACHED 04_relations stores


CACHED 04_relations suppliers


CACHED 04_relations support_tickets


{
  "relations": 31
}


name,source_class_id,target_class_id,cardinality,description
orderItemToProduct,class:order_items,class:products,"{""source_to_target"": {""min"": 1, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Mỗi OrderItem tham chiếu đến một Product qua cột product_id.
orderChannel,class:orders,class:channels,"{""source_to_target"": {""min"": 1, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Mỗi Order tham chiếu đến một Channel qua cột channel_id.
orderItemToPromotion,class:order_items,class:promotions,"{""source_to_target"": {""min"": 0, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Mỗi OrderItem có thể tham chiếu đến một Promotion qua cột promotion_id.
supplierRegion,class:suppliers,class:regions,"{""source_to_target"": {""min"": 1, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Bản ghi Supplier tham chiếu bản ghi Region qua cột region_id.
chatCustomer,class:chats,class:customers,"{""source_to_target"": {""min"": 1, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Bản ghi Chat tham chiếu bản ghi Customer qua cột customer_id.
productBrand,class:products,class:brands,"{""source_to_target"": {""min"": 0, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Bản ghi Product có thể tham chiếu bản ghi Brand qua cột brand_id.
chatChannel,class:chats,class:channels,"{""source_to_target"": {""min"": 0, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Bản ghi Chat có thể tham chiếu bản ghi Channel qua cột channel_id.
agentStore,class:agents,class:stores,"{""source_to_target"": {""min"": 0, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Agent có thể tham chiếu đến một Store qua cột store_id.
addressToCustomer,class:addresses,class:customers,"{""source_to_target"": {""min"": 1, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Bản ghi Address tham chiếu bản ghi Customer qua cột customer_id.
orderStore,class:orders,class:stores,"{""source_to_target"": {""min"": 1, ""max"": 1}, ""target_to_source"": {""min"": 0, ""max"": ""many""}}",Mỗi Order tham chiếu đến một Store qua cột store_id.


## Bước 5 — kiểm định, review ngữ nghĩa và xuất JSON

**Input:** inventory, bases, proposals, contexts, relations được copy vào input bước 5.
**Output:** ontology toàn database, ontology từng Class, QC report, review từng bảng và run summary.
Kiểm định tái dựng kết quả dự kiến từ input để phát hiện artifact bị sửa hoặc stale.
AI reviewer đối chiếu mô tả với schema và kế hoạch Method; review không phải chứng minh tuyệt đối.
Methods vẫn là đề xuất chỉ đọc, chưa chạy SQL. Human review vẫn cần trước khi dùng production.

Traces sinh Base/Relations được copy vào `input/generation_traces/`, để phần QC/tóm tắt chỉ đọc input/output của bước 5. Manifest checksum kiểm kê toàn bộ bộ dữ liệu để phục vụ bàn giao.


In [12]:
def prepare_validation_inputs():
    copy_file(S1 / "output/schema_inventory.json", S5 / "input/schema_inventory.json")
    copy_file(S4 / "output/relations.json", S5 / "input/relations.json")
    for table in read_json(S5 / "input/schema_inventory.json")["tables"]:
        name = table["name"]
        copy_file(S2 / "output/base_ontologies" / f"{name}.json", S5 / "input/base_ontologies" / f"{name}.json")
        copy_file(S2 / "output/proposals" / f"{name}.json", S5 / "input/proposals" / f"{name}.json")
        copy_file(S2 / "output/traces" / f"{name}.json", S5 / "input/generation_traces/base" / f"{name}.json")
        if table["foreign_keys"]:
            copy_file(S4 / "output/traces" / f"{name}.json", S5 / "input/generation_traces/relations" / f"{name}.json")
        copy_file(S3 / "output/reference_contexts" / f"{name}.json", S5 / "input/reference_contexts" / f"{name}.json")
    copy_file(S2 / "input/response_contract.json", S5 / "input/base_response_contract.json")
    copy_file(S4 / "input/response_contract.json", S5 / "input/relation_response_contract.json")

def structural_report(inventory, bases, proposals, contexts, relations):
    findings = []
    tables = {t["name"]: t for t in inventory["tables"]}
    payload = {"database_id": inventory["database_id"], "tables": inventory["tables"]}
    if inventory["schema_fingerprint"] != fingerprint(payload):
        findings.append("Inventory fingerprint không khớp nội dung")
    base_contract = read_json(S5 / "input/base_response_contract.json")
    for name, table in tables.items():
        contract_findings = schema_errors(base_contract, proposals[name])
        if contract_findings:
            findings.extend(f"{name}: {e}" for e in contract_findings)
            continue
        findings.extend(f"{name}: {e}" for e in base_proposal_errors(proposals[name], table))
        expected = assemble_base(table, proposals[name], inventory["database_id"])
        if bases[name] != expected:
            findings.append(f"Base ontology bị sửa/stale: {name}")
        if contexts[name] != make_context(name, inventory, bases):
            findings.append(f"Context bị sửa/stale: {name}")
    expected_relations = {fk["relation_id"]: physical_relation(fk, tables)
                          for t in tables.values() for fk in t["foreign_keys"]}
    ids = [r["relation_id"] for r in relations]
    if not unique(ids) or set(ids) != set(expected_relations):
        findings.append("Relation coverage/IDs không khớp các FK")
    annotations = [{key: r.get(key) for key in ("relation_id", "name", "description", "confidence", "assumptions")}
                   for r in relations]
    findings.extend(schema_errors(read_json(S5 / "input/relation_response_contract.json"), {"relations": annotations}))
    for relation in relations:
        expected = expected_relations.get(relation["relation_id"], {})
        if any(relation.get(key) != value for key, value in expected.items()):
            findings.append(f"Relation endpoints/cardinality bị sửa: {relation['relation_id']}")
    if not unique([b["class"]["name"] for b in bases.values()]):
        findings.append("Tên Class bị trùng")
    return {"structural_status": "PASS" if not findings else "FAIL", "findings": findings,
            "counts": {"classes": len(bases), "properties": sum(len(b["properties"]) for b in bases.values()),
                       "methods": sum(len(b["methods"]) for b in bases.values()), "relations": len(relations)},
            "runtime_readiness": "NOT_IMPLEMENTED", "human_review_status": "PENDING",
            "business_semantics": "PROPOSED_NOT_PROVEN"}

def review_errors(result, table_name):
    errors = []  # Contract đã được kiểm tra trước khi gọi hàm này.
    if result["table_name"] != table_name:
        errors.append("Reviewer trả sai table_name")
    return errors

def validate_and_export():
    inventory = read_json(S5 / "input/schema_inventory.json")
    names = [t["name"] for t in inventory["tables"]]
    bases = {name: read_json(S5 / "input/base_ontologies" / f"{name}.json") for name in names}
    proposals = {name: read_json(S5 / "input/proposals" / f"{name}.json") for name in names}
    contexts = {name: read_json(S5 / "input/reference_contexts" / f"{name}.json") for name in names}
    relation_file = read_json(S5 / "input/relations.json")
    relations = relation_file["relations"]
    report = structural_report(inventory, bases, proposals, contexts, relations)
    if relation_file["schema_fingerprint"] != inventory["schema_fingerprint"]:
        report["findings"].append("Relations sinh từ schema cũ")
        report["structural_status"] = "FAIL"
    write_json(S5 / "output/validation_report.json", report)
    require(report["structural_status"] == "PASS", str(report["findings"]))
    reviews = []
    if RUN_AI_REVIEW:
        for table in inventory["tables"]:
            name = table["name"]
            related = [r for r in relations if f"class:{name}" in (r["source_class_id"], r["target_class_id"])]
            request = {"table": table, "base": bases[name], "relations": related}
            review = ask_llm(S5, name, request, lambda result: review_errors(result, name))
            write_json(S5 / "output/semantic_reviews" / f"{name}.json", review)
            reviews.append(review)
    review_findings = [{"table": r["table_name"], **f} for r in reviews for f in r["findings"]]
    report["ai_review_status"] = "COMPLETED" if RUN_AI_REVIEW else "NOT_RUN"
    report["ai_review_findings"] = review_findings
    report["semantic_status"] = "REVIEW_REQUIRED" if review_findings else "AI_REVIEWED_WITH_LIMITATIONS" if RUN_AI_REVIEW else "NOT_REVIEWED"
    write_json(S5 / "output/validation_report.json", report)
    ontology = {"database_id": inventory["database_id"], "schema_fingerprint": inventory["schema_fingerprint"],
                "classes": [b["class"] for b in bases.values()],
                "properties": [p for b in bases.values() for p in b["properties"]],
                "methods": [m for b in bases.values() for m in b["methods"]], "relations": relations,
                "semantic_status": report["semantic_status"], "human_review_status": "PENDING",
                "runtime_readiness": "NOT_IMPLEMENTED"}
    write_json(S5 / "output/ontology.json", ontology)
    require(read_json(S5 / "output/ontology.json") == ontology, "Ontology export không khớp")
    for name, base in bases.items():
        full = {**base, "relations": [r for r in relations if f"class:{name}" in (r["source_class_id"], r["target_class_id"])]}
        path = S5 / "output/class_ontologies" / f"{name}.json"
        write_json(path, full)
        require(read_json(path) == full, f"Class export không khớp: {name}")
    return report


In [13]:
prepare_validation_inputs()
report = validate_and_export()
show(report)


CACHED 05_validation addresses


CACHED 05_validation agents


CACHED 05_validation brands


CALL 05_validation categories


CACHED 05_validation channels


...

CACHED 05_validation shipments


CACHED 05_validation stores


CACHED 05_validation suppliers


CACHED 05_validation support_tickets


{
  "structural_status": "PASS",
  "findings": [],
  "counts": {
    "classes": 20,
    "properties": 157,
    "methods": 82,
    "relations": 31
  },
  "runtime_readiness": "NOT_IMPLEMENTED",
  "human_review_status": "PENDING",
  "business_semantics": "PROPOSED_NOT_PROVEN",
  "ai_review_status": "COMPLETED",
  "ai_review_findings": [],
  "semantic_status": "AI_REVIEWED_WITH_LIMITATIONS"
}


## Kiểm thử các trường hợp dễ sai

Cell này không gọi API. Đọc schema fixture trong `05_validation/input/qa_cases.sql`,
kiểm tra FK ghép/self/partial UNIQUE/expression UNIQUE/nullable và các Method sai có chủ đích.
Sau đó sửa bản sao artifacts trong RAM để chứng minh validator phát hiện sai lệch.
Không dùng fixture để thay response LLM; không ghi fixture thành ontology thật.

In [14]:
def run_regression_checks():
    checks = []

    def record(name, passed):
        require(passed, "Regression FAILED: " + name)
        checks.append({"check": name, "status": "PASS"})

    with closing(sqlite3.connect(":memory:")) as connection:
        connection.executescript((S5 / "input/qa_cases.sql").read_text(encoding="utf-8"))
        connection.execute("INSERT INTO nodes(id,parent_id) VALUES(1,1)")
        fixture = {t["name"]: t for t in read_schema(connection, {})}
        record("Self-FK cho phép tham chiếu chính bản ghi", not connection.execute("PRAGMA foreign_key_check").fetchall())
        described = read_schema(connection, {"tables": {"nodes": {"description": "Cấu trúc nút", "columns": {"parent_id": {"description": None}}}}})
    record("Metadata làm thay đổi fingerprint", fingerprint(list(fixture.values())) != fingerprint(described))
    record("Metadata null chuẩn hóa thành chuỗi rỗng", next(t for t in described if t["name"] == "nodes")["columns"][1]["description"] == "")
    record("TEXT PRIMARY KEY rowid vẫn nullable", fixture["nullable_pk"]["columns"][0]["nullable"])
    record("INTEGER PRIMARY KEY không nullable", not fixture["nodes"]["columns"][0]["nullable"])
    record("STRICT PRIMARY KEY không nullable", not fixture["strict_pk"]["columns"][0]["nullable"])
    record("WITHOUT ROWID PRIMARY KEY không nullable", not fixture["without_rowid"]["columns"][0]["nullable"])
    record("Generated column được giữ", fixture["nullable_pk"]["columns"][2]["generated"])
    fk = fixture["child_unique"]["foreign_keys"][0]
    record("FK ghép suy ra đủ PK đích", fk["target_columns"] == ["a", "b"])
    relation = physical_relation(fk, fixture)
    record("FK nullable có min=0", relation["cardinality"]["source_to_target"]["min"] == 0)
    record("FK ghép UNIQUE có max ngược=1", relation["cardinality"]["target_to_source"]["max"] == 1)
    partial = physical_relation(fixture["child_partial"]["foreign_keys"][0], fixture)
    record("Partial UNIQUE không tạo 1-1 toàn bảng", partial["cardinality"]["target_to_source"]["max"] == "many")
    record("FK NOT NULL có min=1", partial["cardinality"]["source_to_target"]["min"] == 1)
    record("Expression UNIQUE không thành khóa cột", not has_unique_key(fixture["expressions"], ["label"]))
    record("FK khác hoa/thường được chuẩn hóa", fixture["case_child"]["foreign_keys"][0]["target_table"] == "nodes")
    duplicate_ids = [fk["relation_id"] for fk in fixture["repeated_fk"]["foreign_keys"]]
    record("FK trùng nội dung vẫn có IDs riêng", len(duplicate_ids) == 2 and unique(duplicate_ids))

    lookup = {
        "name": "lookupNode", "description": "Tra cứu nút theo khóa", "operation": "lookup",
        "input_parameters": [{"name": "id", "type": "integer", "required": True,
                              "source_column": "id", "operator": "eq"}],
        "group_by": [], "output": {"kind": "object", "source_columns": ["id"], "aggregations": []},
        "source_columns": ["id"], "confidence": 0.5,
    }
    record("Lookup equality theo PK được chấp nhận", not method_errors(lookup, fixture["nodes"]))
    wrong = copy.deepcopy(lookup)
    wrong["input_parameters"][0]["operator"] = "gte"
    record("Lookup range theo PK bị chặn", any("equality" in e for e in method_errors(wrong, fixture["nodes"])))
    wrong = copy.deepcopy(lookup)
    wrong["input_parameters"][0]["required"] = False
    record("Lookup thiếu required key bị chặn", any("equality" in e for e in method_errors(wrong, fixture["nodes"])))
    wrong = copy.deepcopy(lookup)
    wrong["source_columns"].append("parent_id")
    record("Cột khai báo dư bị chặn", any("hợp chính xác" in e for e in method_errors(wrong, fixture["nodes"])))
    wrong = copy.deepcopy(lookup)
    wrong["output"]["source_columns"] = ["other_table.id"]
    record("Method ngoài bảng bị chặn", any("ngoài bảng" in e for e in method_errors(wrong, fixture["nodes"])))
    wrong = copy.deepcopy(lookup)
    wrong["input_parameters"][0]["type"] = "string"
    record("Sai parameter type bị chặn", any("Sai type" in e for e in method_errors(wrong, fixture["nodes"])))
    grouped = {
        "name": "countByParent", "description": "Đếm nút theo parent", "operation": "aggregate",
        "input_parameters": [], "group_by": ["parent_id"],
        "output": {"kind": "rows", "source_columns": ["parent_id"],
                   "aggregations": [{"name": "rowCount", "function": "count", "source_column": None}]},
        "source_columns": ["parent_id"], "confidence": 0.5,
    }
    record("Aggregate có group đúng được chấp nhận", not method_errors(grouped, fixture["nodes"]))
    wrong = copy.deepcopy(grouped)
    wrong["output"]["source_columns"] = []
    record("Aggregate thiếu cột nhóm bị chặn", any("cột nhóm" in e for e in method_errors(wrong, fixture["nodes"])))
    wrong = copy.deepcopy(grouped)
    wrong["output"]["kind"] = "scalar"
    record("Aggregate nhóm trả scalar bị chặn", any("trả rows" in e for e in method_errors(wrong, fixture["nodes"])))
    wrong = copy.deepcopy(grouped)
    wrong["group_by"] = []
    wrong["output"] = {"kind": "scalar", "source_columns": [],
                       "aggregations": [{"name": "sumId", "function": "sum", "source_column": "id"}]}
    wrong["source_columns"] = ["id"]
    record("SUM trên TEXT bị chặn", any("cột số" in e for e in method_errors(wrong, fixture["nullable_pk"])))

    inventory = read_json(S5 / "input/schema_inventory.json")
    names = [t["name"] for t in inventory["tables"]]
    bases = {n: read_json(S5 / "input/base_ontologies" / f"{n}.json") for n in names}
    proposals = {n: read_json(S5 / "input/proposals" / f"{n}.json") for n in names}
    contexts = {n: read_json(S5 / "input/reference_contexts" / f"{n}.json") for n in names}
    relations = read_json(S5 / "input/relations.json")["relations"]
    record("Artifacts thật qua structural validator", structural_report(inventory, bases, proposals, contexts, relations)["structural_status"] == "PASS")
    wrong_bases = copy.deepcopy(bases)
    wrong_bases["products"]["properties"][0]["nullable"] = not wrong_bases["products"]["properties"][0]["nullable"]
    record("Property vật lý bị sửa được phát hiện", structural_report(inventory, wrong_bases, proposals, contexts, relations)["structural_status"] == "FAIL")
    wrong_relations = copy.deepcopy(relations)
    wrong_relations[0]["cardinality"]["source_to_target"]["max"] = "many"
    record("Cardinality bị sửa được phát hiện", structural_report(inventory, bases, proposals, contexts, wrong_relations)["structural_status"] == "FAIL")
    wrong_contexts = copy.deepcopy(contexts)
    wrong_contexts["products"]["references"] = []
    record("Context bị sửa được phát hiện", structural_report(inventory, bases, proposals, wrong_contexts, relations)["structural_status"] == "FAIL")
    record("Relation thiếu được phát hiện", structural_report(inventory, bases, proposals, contexts, relations[1:])["structural_status"] == "FAIL")
    self_fk = next(fk for t in inventory["tables"] if t["name"] == "categories" for fk in t["foreign_keys"])
    record("Self-FK chỉ có một Relation toàn cục", sum(r["relation_id"] == self_fk["relation_id"] for r in relations) == 1)
    record("Context self không tự lặp Class", all(r["class_id"] != "class:categories" for r in contexts["categories"]["references"]))
    local_contract = read_json(S5 / "input/base_response_contract.json")
    original_contract = copy.deepcopy(local_contract)
    grammar_contract = generation_contract(local_contract)
    record("API grammar bỏ uniqueItems không được hỗ trợ", '"uniqueItems"' not in json.dumps(grammar_contract))
    record("Contract local vẫn giữ uniqueItems", '"uniqueItems"' in json.dumps(local_contract))
    record("Chuyển schema API không sửa contract local", local_contract == original_contract)
    record("Validator local bắt array trùng", bool(schema_errors({"type":"array", "uniqueItems":True}, ["id", "id"])))
    category_table = next(t for t in inventory["tables"] if t["name"] == "categories")
    wrong_category = copy.deepcopy(proposals["categories"])
    wrong_category["description"] = "Danh mục tham chiếu danh mục khác"
    record("Mô tả self-FK không loại bỏ tự tham chiếu", any("self-FK" in e for e in base_proposal_errors(wrong_category, category_table)))
    result = {"status": "PASS", "checks": len(checks), "results": checks}
    write_json(S5 / "output/regression_report.json", result)
    return result

regression = run_regression_checks()
show_table(regression["results"], ["check", "status"])
print("Regression checks:", regression["checks"])

check,status
Self-FK cho phép tham chiếu chính bản ghi,PASS
Metadata làm thay đổi fingerprint,PASS
Metadata null chuẩn hóa thành chuỗi rỗng,PASS
TEXT PRIMARY KEY rowid vẫn nullable,PASS
INTEGER PRIMARY KEY không nullable,PASS
STRICT PRIMARY KEY không nullable,PASS
WITHOUT ROWID PRIMARY KEY không nullable,PASS
Generated column được giữ,PASS
FK ghép suy ra đủ PK đích,PASS
FK nullable có min=0,PASS


Regression checks: 38


## Tóm tắt lần chạy và kiểm tra trace thinking

`reasoning_content`/reasoning_tokens là bằng chứng server trả thinking. Không chỉ kiểm tra flag cấu hình.
Cache lưu trace của lần gọi trước; thời gian/usage là của các HTTP attempts đã lưu, không phải đo lại khi replay.


In [15]:
def summarize_run():
    calls = []
    trace_groups = [
        ("02_base_ontology", S5 / "input/generation_traces/base"),
        ("04_relations", S5 / "input/generation_traces/relations"),
        ("05_validation", S5 / "output/traces"),
    ]
    for step_name, trace_folder in trace_groups:
        for path in sorted(trace_folder.glob("*.json")):
            trace = read_json(path)
            for attempt in trace["attempts"]:
                usage = attempt.get("usage", {})
                details = usage.get("completion_tokens_details") or {}
                calls.append({"step": step_name, "table": path.stem, "attempt": attempt["attempt"],
                              "finish_reason": attempt.get("finish_reason"),
                              "structured_output": trace.get("request", {}).get("response_format", {}).get("type"),
                              "reasoning_characters": len(attempt.get("reasoning_content", "")),
                              "reasoning_tokens": details.get("reasoning_tokens", 0),
                              "total_tokens": usage.get("total_tokens", 0),
                              "elapsed_seconds": attempt.get("elapsed_seconds", 0),
                              "validation_error": attempt.get("validation_error", "")})
    summary = {"completed_at_utc": datetime.now(timezone.utc).isoformat(),
               "model": settings["model"], "thinking_requested": settings["enable_thinking"],
               "thinking_observed_calls": sum(c["reasoning_characters"] > 0 or c["reasoning_tokens"] > 0 for c in calls),
               "http_attempts": len(calls), "total_tokens": sum(c["total_tokens"] for c in calls),
               "counts": read_json(S5 / "output/validation_report.json")["counts"], "calls": calls}
    write_json(S5 / "output/run_summary.json", summary)
    manifest = {"model": settings["model"], "thinking": settings["enable_thinking"],
                "python": os.sys.version.split()[0], "sqlite": sqlite3.sqlite_version,
                "files": {}}
    for step in (S1, S2, S3, S4, S5):
        for path in sorted(step.rglob("*")):
            if path.is_file() and path.name != "reproducibility_manifest.json" and not path.name.endswith(".local.sqlite3"):
                manifest["files"][str(path.relative_to(DATA))] = file_hash(path)
    write_json(S5 / "output/reproducibility_manifest.json", manifest)
    return summary

summary = summarize_run()
show({k: v for k, v in summary.items() if k != "calls"})
show_table(summary["calls"], ["step", "table", "attempt", "reasoning_tokens", "finish_reason", "validation_error"])


{
  "completed_at_utc": "2026-10-26T08:13:25.215669+00:00",
  "model": "qwen/qwen3.6-27b",
  "thinking_requested": true,
  "thinking_observed_calls": 58,
  "http_attempts": 58,
  "total_tokens": 352803,
  "counts": {
    "classes": 20,
    "properties": 157,
    "methods": 82,
    "relations": 31
  }
}


step,table,attempt,reasoning_tokens,finish_reason,validation_error
02_base_ontology,addresses,1,2516,stop,
02_base_ontology,agents,1,803,stop,
02_base_ontology,brands,1,567,stop,Không dùng khẳng định tự tăng khi DDL không khai báo AUTOINCREMENT; mô tả là khóa chính INTEGER
02_base_ontology,brands,2,336,stop,
02_base_ontology,categories,1,876,stop,
02_base_ontology,channels,1,601,stop,
02_base_ontology,chats,1,1686,stop,
02_base_ontology,customers,1,987,stop,
02_base_ontology,inventory_snapshots,1,1719,stop,
02_base_ontology,order_items,1,1540,stop,


## Cách tự sửa và chạy lại

- Sửa prompt/contract ngay trong `02_base_ontology/input/` hoặc `04_relations/input/`; chạy lại từ bước đó xuống.
- Sửa mô tả nguồn tại `01_schema/input/metadata.json`; chạy lại từ bước 1.
- Thay database: đặt SQLite vào `01_schema/input/database.local.sqlite3`. Không commit rows lên Git.
- Trên server chỉ có schema.sql: vẫn trích xuất và chạy LLM được; kiểm tra FK trên dữ liệu thật cần cung cấp SQLite thật.
- API key mới: dùng biến môi trường hoặc `secrets.local.json`; notebook không cần biết file YAML bên ngoài.
- Đổi thinking/model trong settings.json, chạy lại cell thiết lập rồi từ bước 2. Muốn giữ hai lần chạy,
  copy cả notebook + folder sang thư mục mới trước khi đổi; output hiện tại được ghi lại.
- JSON sai: xem `output/traces/<bảng>.json`, trường validation_error. HTTP/timeout: sửa kết nối/settings rồi chạy lại cell lỗi.
- Chỉ cần đọc kết quả: mở `05_validation/output/ontology.json`, `class_ontologies/products.json`, `validation_report.json`.

PASS cấu trúc không chứng minh mọi business meaning đúng. AI reviewer dùng cùng model nên có thể cùng mắc lỗi;
hãy đối chiếu nguồn và QC_REVIEW.md trước khi duyệt nghiệp vụ hoặc triển khai SQL.
